In [1]:
import pandas as pd

## Implementing the retrieval metrics


In [12]:
def precision(k:int, q_rels:list, q_ret:list) -> float:
    cpt = 0
    for doc in q_ret[:k]:
        if doc in q_rels:
            cpt += 1
    return cpt / k

def recall(k:int, q_rels:list, q_ret:list) -> float:
    cpt = 0
    for doc in q_ret[:k]:
        if doc in q_rels:
            cpt += 1
    return cpt / len(q_rels) if q_rels else 0

## Connect to the VDB

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "hybrid_collection"

def retrieve_relevant_chunks(query: str, k: int, retrieval_mode: RetrievalMode = RetrievalMode.HYBRID, use_reranker:str = False) -> list:
    # Import embedding model
    dense_embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25") # Using the qdrant bm25 model for sparse embeddings

    vector_store = QdrantVectorStore.from_existing_collection(
        embedding=dense_embeddings, #embeddings uses the dense embedding (classical embedding that we know)
        sparse_embedding=sparse_embeddings,  #sparse embeddings mean the bm25
        retrieval_mode=retrieval_mode, # Select the retrieval mode : DENSE / HYBRID / SPARSE
        url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
    )

    documents = vector_store.similarity_search(
        query = query,
        k=k
    )

    chunks_ids = []
    for chunk in documents:
        chunks_ids.append(chunk.metadata.get("_id"))
        
    return chunks_ids

In [5]:
import json

benchmark_questions = pd.read_json("../questions.jsonl", lines=True)
benchmark_questions.index = benchmark_questions.query_id

with open("../qrels.json", "r", encoding="utf-8") as file:
    qrels = json.load(file)

In [ ]:
K = 5

question_ids = benchmark_questions.index.tolist()
number_of_questions = len(question_ids)

retrieval_strategies = [RetrievalMode.DENSE, RetrievalMode.SPARSE, RetrievalMode.HYBRID]

for retrieval_strategy in retrieval_strategies:
    average_precision = 0
    average_recall = 0
    print(f"\n\nRetrieval strategy: {retrieval_strategy}")
    for q_id in question_ids:

        query = benchmark_questions.loc[q_id].question
        q_rels = qrels[q_id]

        q_ret = retrieve_relevant_chunks(query, K, retrieval_mode=retrieval_strategy)

        precisionAtK = precision(K, q_rels, q_ret)
        recallAtK = recall(K, q_rels, q_ret)


        average_precision += precisionAtK
        average_recall += recallAtK
        

    print(f' precision : {average_precision / number_of_questions}, recall :  {average_recall / number_of_questions}')



Retrieval strategy: RetrievalMode.DENSE


In [ ]:
K = 5

question_ids = benchmark_questions.index.tolist()
number_of_questions = len(question_ids)

average_precision = 0
average_recall = 0
for q_id in question_ids:

    query = benchmark_questions.loc[q_id].question
    q_rels = qrels[q_id]

    q_ret = retrieve_relevant_chunks(query, K, retrieval_mode=RetrievalMode.DENSE)

    precisionAtK = precision(K, q_rels, q_ret)
    recallAtK = recall(K, q_rels, q_ret)


    average_precision += precisionAtK
    average_recall += recallAtK
    

print(f' precision : {average_precision / number_of_questions}, recall :  {average_recall / number_of_questions}')